# Recoverability Study — GPU Battery Runner (Colab)

Runs the validity-gate + reproduction battery on a CUDA GPU. The scripts auto-detect CUDA (no code changes needed).

**Steps:** set runtime to GPU (Runtime → Change runtime type → A100/L4), then run cells top to bottom.
Upload `recoverability_code.zip` when prompted. Results are zipped for download at the end.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set Runtime to GPU')

In [ ]:
# Upload recoverability_code.zip (from your project: recoverability_study/colab/)
from google.colab import files
up = files.upload()
!rm -rf code && mkdir -p code && unzip -o recoverability_code.zip -d code >/dev/null && ls code

In [ ]:
%cd code
# Validity gate: generators + oracles
!python verify_generators.py

In [ ]:
# Reproduction 1: Fibonacci / modular vs n-gram (fast)
!python exp5_baselines.py

In [ ]:
# Reproduction 2: elementary CA FULL SCALE (Rule 30 + Rule 90). Rule 90 is the
# parity-class rule that needs full scale. Includes free-run rollout eval.
!python exp_ca.py --rules 30 90 --train_seeds 2000 --test_seeds 500 --steps 32 --steps_ood 64 --density_test_list 0.2 0.8

In [ ]:
# Reproduction 3: parity CA, held-out neighborhoods (full scale)
!python exp_parity_ca.py --width 64 --a_fraction 0.6 --n_train 5000 --n_test 1000 --epochs 50

In [ ]:
# Reproduction 4: Dyck-1 / Dyck-2 FULL SCALE (10000 train / 2000 test)
!python exp_dyck.py

In [ ]:
# Package all results for download
!zip -r /content/battery_results.zip runs >/dev/null 2>&1; echo done
from google.colab import files
files.download('/content/battery_results.zip')